# Ajuste fino LoRA de Stable Diffusion con `diffusers`

Esta práctica adapta Stable Diffusion v1.5 mediante LoRA usando una muestra de Fashion-MNIST. El conjunto tiene imágenes pequeñas de diez categorías de prendas; el cuaderno las convierte a RGB y les asigna descripciones breves a partir de sus etiquetas. La comparación usa el mismo prompt y la misma semilla en el modelo base y el adaptador.

**Alcance:** experimento educativo de 200 pasos en Colab con GPU. No se busca producir imágenes listas para catálogo. El checkpoint público se descarga sin token. El modelo usa licencia CreativeML OpenRAIL-M; revise sus condiciones antes de redistribuir pesos o imágenes.


In [ ]:
!nvidia-smi
!pip -q install "diffusers==0.35.1" "transformers==4.57.1" "huggingface_hub>=0.34,<1.0" "datasets<4" "accelerate>=1.0" "peft>=0.17"
!rm -rf /content/diffusers
!git clone --depth 1 --branch v0.35.1 https://github.com/huggingface/diffusers.git /content/diffusers
!pip -q install --no-deps -e /content/diffusers


In [ ]:
import os, torch, matplotlib.pyplot as plt

if not torch.cuda.is_available():
    raise RuntimeError('Activa un entorno GPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución.')
print('GPU:', torch.cuda.get_device_name(0))


## 1. Datos y descripciones

Fashion-MNIST contiene imágenes de prendas en diez clases. Para que el script de Diffusers reciba pares imagen-texto, se convierten las etiquetas del conjunto en frases descriptivas sencillas y se guarda una muestra de 500 imágenes como ImageFolder. Las imágenes originales son monocromas de 28 × 28 píxeles; se pasan a RGB y el script las ajusta a 256 × 256. Las frases no son anotaciones humanas: describen solo la categoría, por lo que la práctica permite observar el flujo de LoRA, no aprender detalles finos de diseño.


In [ ]:
from torchvision.datasets import FashionMNIST
from torchvision.transforms.functional import to_pil_image
from pathlib import Path
import json, shutil

DATASET='/content/fashion_lora_data'
root=Path(DATASET); root.mkdir(parents=True, exist_ok=True)
for f in root.glob('*.png'): f.unlink()
labels=['camiseta','pantalón','suéter','vestido','abrigo','sandalia','camisa','zapatilla','bolso','bota']
base=FashionMNIST('/content/fashion_mnist', train=True, download=True)
# Selección balanceada y determinista: 50 imágenes por cada clase.
counts={i:0 for i in range(10)}; rows=[]
for idx,(image,label) in enumerate(base):
    if counts[label] >= 50: continue
    rgb=image.convert('RGB').resize((256,256))
    name=f'{len(rows):04d}.png'; rgb.save(root/name)
    caption=f'a small grayscale catalog image of a {labels[label]} on a plain background'
    rows.append({'file_name':name,'text':caption}); counts[label]+=1
    if len(rows)==500: break
with open(root/'metadata.jsonl','w',encoding='utf-8') as f:
    for row in rows: f.write(json.dumps(row,ensure_ascii=False)+'\n')
print(f'Imágenes preparadas: {len(rows)} | distribución: {counts}')
for row in rows[:3]:
    print(row['text']); display(__import__('PIL').Image.open(root/row['file_name']))


## 2. Modelo base y parámetros

Se utiliza el checkpoint `stable-diffusion-v1-5/stable-diffusion-v1-5` y el script oficial `train_text_to_image_lora.py` de Diffusers. LoRA actualiza matrices de bajo rango en las capas de atención, mientras mantiene congelados la mayoría de los parámetros del modelo. La configuración de 200 pasos, resolución de 256, lote 1 y precisión FP16 reduce el uso de memoria en Colab; los resultados son exploratorios y dependen de la GPU y del rendimiento de la GPU.


In [ ]:
MODEL='stable-diffusion-v1-5/stable-diffusion-v1-5'
DATASET='/content/fashion_lora_data'
OUTPUT='/content/carlos-lora-fashion'
SCRIPT='/content/diffusers/examples/text_to_image/train_text_to_image_lora.py'
assert os.path.exists(SCRIPT), 'No se encontró el script oficial de entrenamiento'
assert os.path.exists(os.path.join(DATASET,'metadata.jsonl')), 'No se prepararon las imágenes y captions'


In [ ]:
!accelerate launch $SCRIPT \
  --pretrained_model_name_or_path=$MODEL \
  --train_data_dir=$DATASET \
  --image_column=image \
  --caption_column=text \
  --resolution=256 \
  --center_crop --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --learning_rate=1e-4 \
  --lr_scheduler=constant \
  --lr_warmup_steps=0 \
  --max_train_steps=200 \
  --checkpointing_steps=100 \
  --validation_prompt='a small grayscale catalog image of a sneaker on a plain background' \
  --num_validation_images=2 \
  --mixed_precision=fp16 \
  --output_dir=$OUTPUT \
  --seed=42


## 3. Comparación antes y después

La evaluación mantiene la frase, la semilla, el modelo base y el número de pasos de inferencia. Se guardan ambas salidas y una comparación lado a lado. La revisión es visual y cualitativa; no se afirma que el adaptador mejore el modelo sin una métrica y una evaluación más amplia.


In [ ]:
from diffusers import StableDiffusionPipeline
from pathlib import Path

device='cuda'; dtype=torch.float16
prompt='a small grayscale catalog image of a sneaker on a plain background'
base=StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=dtype, safety_checker=None).to(device)
base.set_progress_bar_config(disable=True)
base_image=base(prompt, num_inference_steps=25, guidance_scale=7.0, generator=torch.Generator(device=device).manual_seed(42)).images[0]
lora=StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=dtype, safety_checker=None).to(device)
lora.load_lora_weights(OUTPUT)
lora.set_progress_bar_config(disable=True)
lora_image=lora(prompt, num_inference_steps=25, guidance_scale=7.0, generator=torch.Generator(device=device).manual_seed(42)).images[0]
os.makedirs('/content/resultados_lora_carlos', exist_ok=True)
base_image.save('/content/resultados_lora_carlos/base.png'); lora_image.save('/content/resultados_lora_carlos/lora.png')
fig, ax=plt.subplots(1,2,figsize=(10,5)); ax[0].imshow(base_image); ax[0].set_title('Stable Diffusion base'); ax[1].imshow(lora_image); ax[1].set_title('Base + LoRA');
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig('/content/resultados_lora_carlos/comparacion.png',dpi=160); plt.show()


## 4. Registro y límites

Registre la GPU, número real de pasos completados, tiempo, pérdida final si aparece en la consola y observaciones de las imágenes. Una muestra de 500 imágenes y 200 pasos puede causar sobreajuste o dejar las salidas poco definidas. El dataset tiene diez clases, pero las descripciones se generan automáticamente desde la etiqueta y no contienen atributos visuales detallados.


In [ ]:
import json, datetime
registro={'fecha_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'gpu':torch.cuda.get_device_name(0),'base_model':MODEL,'dataset':'Fashion-MNIST (muestra balanceada de 500 imágenes)','resolution':256,'steps_requested':200,'seed':42,'prompt':prompt,'observacion':'Completar tras revisar base.png, lora.png y las pérdidas reportadas por el entrenamiento.'}
with open('/content/resultados_lora_carlos/registro.json','w',encoding='utf-8') as f: json.dump(registro,f,ensure_ascii=False,indent=2)
print(json.dumps(registro,ensure_ascii=False,indent=2))


## Referencias

Hugging Face. (s. f.). *Text-to-image fine-tuning*. Diffusers documentation. https://huggingface.co/docs/diffusers/en/training/text2image

Rombach, R., Blattmann, A., Lorenz, D., Esser, P., & Ommer, B. (2022). High-resolution image synthesis with latent diffusion models. *Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition*, 10684–10695. https://doi.org/10.1109/CVPR52688.2022.01042

Xiao, H., Rasul, K., & Vollgraf, R. (2017). Fashion-MNIST: A novel image dataset for benchmarking machine learning algorithms. *arXiv*. https://arxiv.org/abs/1708.07747

Zalando Research. (2017). *Fashion-MNIST* [Conjunto de datos; licencia MIT]. GitHub. https://github.com/zalandoresearch/fashion-mnist
